In [31]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

In [33]:
torch.manual_seed(seed= 42)

In [34]:
torch.cuda.is_available()

True

In [35]:
device= torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [36]:
device

device(type='cuda')

In [37]:
df= pd.read_csv(r"/content/drive/MyDrive/dataset/fashion-mnist.csv")
df.head()


,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [38]:
x= df.drop(columns= ['label'])
y= df['label']

In [39]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test= train_test_split(x, y, random_state= 42, test_size= 0.2)

In [40]:
from torchvision.transforms import transforms
custom_transforms= transforms.Compose(   #this is basically like a pipeline
[
    transforms.Resize(size= (256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),      #converts PIL image into tensor and automatically scale it from 0.0-1.0
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
]
)

In [41]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image


In [42]:
class CustomDataset(Dataset):

    def __init__(self, features, labels, transform):
        self.features = torch.tensor(
            features.reshape(-1, 1, 28, 28)
        )
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.transform = transform

    def __len__(self):
        return self.labels.shape[0]

    def __getitem__(self, index):  # normally gets one index at a time
        image = self.features[index]

        # Convert to uint8 because we want to convert the tensor to PIL
        image = image.to(dtype=torch.uint8)

        # Convert grayscale (1 channel) to RGB (3 channels)
        image = torch.concatenate([image, image, image])


        #converting it to pil
        image= transforms.ToPILImage()(image)

        #sending it to transformer
        image= self.transform(image)


        return image, self.labels[index]




In [43]:
train_dataset= CustomDataset(x_train.values, y_train.values, custom_transforms)
test_dataset= CustomDataset(x_test.values, y_test.values, custom_transforms)

In [44]:
train_batch= DataLoader(dataset= train_dataset,
                        batch_size= 32,
                        pin_memory= True, shuffle= True)

In [45]:
test_batch= DataLoader(
    dataset= test_dataset,
    batch_size= 32,
    pin_memory= True,
    shuffle= True
)

In [46]:
train_dataset[0][0].shape

torch.Size([3, 224, 224])

In [47]:
train_dataset[0][1]

tensor(5)

In [50]:
#fetch vgg16 model
from torchvision.models import vgg16, VGG16_Weights
model= vgg16(weights= VGG16_Weights.DEFAULT)

In [51]:
model

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [52]:
model.features

Sequential(
  (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU(inplace=True)
  (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (3): ReLU(inplace=True)
  (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (6): ReLU(inplace=True)
  (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (8): ReLU(inplace=True)
  (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (11): ReLU(inplace=True)
  (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (13): ReLU(inplace=True)
  (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (15): ReLU(inplace=True)
  (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (17): Conv2d(256, 512, kernel_si

In [53]:
model.classifier

Sequential(
  (0): Linear(in_features=25088, out_features=4096, bias=True)
  (1): ReLU(inplace=True)
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=4096, out_features=4096, bias=True)
  (4): ReLU(inplace=True)
  (5): Dropout(p=0.5, inplace=False)
  (6): Linear(in_features=4096, out_features=1000, bias=True)
)

In [54]:
# here we need to change only the weights of classification part. So we freeze the feature part
#model.features.parameters() gives the information of the parameters of feature part
for param in model.features.parameters():
    param.requires_grad= False

In [55]:
model.features[0].bias.shape

torch.Size([64])

In [56]:
# we will keeep the feature part as it is and change classification part
model.classifier= nn.Sequential(
    nn.Linear(25088, 128),
                nn.BatchNorm1d(128),
                nn.ReLU(),
                nn.Dropout(p= 0.3),
    
                nn.Linear(128, 64),
                nn.BatchNorm1d(64),
                nn.ReLU(),
                nn.Dropout(p= 0.3),
    
    
                nn.Linear(64, 10)
)

In [57]:
model

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [63]:
learning_rate= 1e-4
epoch= 10
regularization= 0.03

In [64]:
model= model.to(device= device)

In [65]:
critetation= nn.CrossEntropyLoss()
optimizer= optim.Adam(params= model.classifier.parameters(),lr= learning_rate, weight_decay= regularization) 
#we only want to change param of classifier part

In [66]:
model.train()

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [67]:
for i in range(epoch):
    total_loss= 0

    for batch_feature, batch_label in train_batch:
        batch_feature= batch_feature.to(device= device)
        batch_label= batch_label.to(device= device)


        y_pred= model(batch_feature)

        loss= critetation(y_pred, batch_label)

        
        total_loss+= loss.item()

        #reset all gradient
        optimizer.zero_grad()

        #calcn gradient
        loss.backward()

        #update trainable param
        optimizer.step()


    print(f"Epoch: {i+1}; loss: {total_loss/len(train_batch)}")



Epoch: 1; loss: 0.6534512680371602
Epoch: 2; loss: 0.4363068189720313
Epoch: 3; loss: 0.41618966725468637
Epoch: 4; loss: 0.4196913411319256
Epoch: 5; loss: 0.42440956823031106
Epoch: 6; loss: 0.4273320425848166
Epoch: 7; loss: 0.4259357488552729
Epoch: 8; loss: 0.42379139105478925
Epoch: 9; loss: 0.42310613264640173
Epoch: 10; loss: 0.41742814041177434


In [68]:
model.eval()

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [70]:
accurate= 0
num= 0
for batch_features, batch_label in test_batch:
    batch_features= batch_features.to(device= device)
    batch_label= batch_label.to(device= device)
    
    with torch.no_grad():

        y_pred= model(batch_features)

    _, idx= torch.max(y_pred, dim= 1)

    corr= torch.where(
        idx== batch_label,
        1,
        0

    )

    accurate= accurate + corr.sum().item()
    num= num+ batch_label.shape[0]

print("Accuracy", accurate/num)



Accuracy 0.918


In [72]:
#training loss
accurate= 0
num= 0
for batch_features, batch_label in train_batch:
    batch_features= batch_features.to(device= device)
    batch_label= batch_label.to(device= device)
    
    with torch.no_grad():

        y_pred= model(batch_features)

    _, idx= torch.max(y_pred, dim= 1)

    corr= torch.where(
        idx== batch_label,
        1,
        0

    )

    accurate= accurate + corr.sum().item()
    num= num+ batch_label.shape[0]

print("Accuracy", accurate/num)



Accuracy 0.9453958333333333
